In [2]:
import json
from typing import Literal
import timeit

import outlines
from huggingface_hub import snapshot_download
from mlx_lm import load
from pydantic import BaseModel, Field


class AnimalClassification(BaseModel):
    animal: str = Field(description="Animal name provided by the user")
    category: Literal["Mammal", "Bird", "Reptile", "Amphibian", "Fish", "Other"] = Field(
        description="Animal category selected from the fixed options"
    )
    explanation: str = Field(description="Brief reasoning for the chosen category")


def test():
    MODEL_NAME = "mlx-community/Qwen3.5-4B-MLX-4bit"
    animal_name = "Crocodile"
    choices = ["Mammal", "Bird", "Reptile", "Amphibian", "Fish", "Other"]

    # Load a model that's already downloaded locally, avoiding a Hugging Face
    # connection during the demo.
    model_path = snapshot_download(MODEL_NAME, local_files_only=True)
    model, tokenizer = load(model_path)
    qwen = outlines.from_mlxlm(model, tokenizer)

    messages = [
        {
            "role": "system",
            "content": (
                "You are an animal classification assistant. Choose the correct "
                "category from the fixed options provided by the user. "
                "Do not add any category of your own, and keep the explanation short."
            ),
        },
        {
            "role": "user",
            "content": (
                f"Animal: {animal_name}\n"
                f"Options: {json.dumps(choices, ensure_ascii=False)}"
            ),
        },
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,  # Qwen3.5 System 1 模式
    )

    raw = qwen(prompt, output_type=AnimalClassification, max_tokens=256)
    print(raw)
    result = AnimalClassification.model_validate_json(raw)

    print(json.dumps(result.model_dump(), ensure_ascii=False, indent=2))

total_time = timeit.timeit(test, number=1)
print(f"Total time: {total_time:.2f} seconds")

{ "animal": "Crocodile", "category": "Reptile" , "explanation": "Crocodiles are cold-blooded vertebrates with scales, which classifies them as reptiles." }
{
  "animal": "Crocodile",
  "category": "Reptile",
  "explanation": "Crocodiles are cold-blooded vertebrates with scales, which classifies them as reptiles."
}
Total time: 2.85 seconds
